# 00 — setup

**Summary.** Checks that every file these notebooks read is on this machine, and names the command
that builds each one that is not. Nothing here measures anything; it exists so that the other seven
notebooks fail with a sentence instead of a traceback.

The corpus is 20,000 manifest works from three museums, **19,791 distinct images** after the sha256
dedupe, embedded with a local CLIP ViT-B/32. All of it is derived from tracked files
(`corpus/manifest.jsonl`, `aesthetic/influences/*.resolved.json`) plus 3.0GB of gitignored pixels.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import kusama as k

## What is on this machine

In [2]:
checks = [
    ("corpus/manifest.jsonl",            k.CORPUS / "manifest.jsonl",       "tracked — it is the evidence"),
    ("corpus/images/",                   k.IMAGES,                          "npm run corpus -- images"),
    ("corpus/clip.f32",                  k.CORPUS / "clip.f32",             "npm run corpus -- embed"),
    ("corpus/clip-index.json",           k.CORPUS / "clip-index.json",      "npm run corpus -- embed"),
    ("corpus/analytics/manifest.csv",    k.ANALYTICS / "manifest.csv",      "npm run corpus -- export-analytics"),
    ("corpus/analytics/knn-k20.csv",     k.ANALYTICS / "knn-k20.csv",       "npm run corpus -- export-analytics"),
    ("corpus/analytics/band.json",       k.ANALYTICS / "band.json",         "npm run corpus -- export-analytics"),
    ("aesthetic/influences/",            k.INFLUENCES,                      "npm run corpus -- influences resolve"),
    ("dist/studio/corpus.js",            k.ROOT / "dist/studio/corpus.js",  "npm run build"),
]
for name, path, how in checks:
    print(f"{'OK  ' if path.exists() else 'MISSING'}  {name:<34} {'' if path.exists() else how}")

OK    corpus/manifest.jsonl              
OK    corpus/images/                     
OK    corpus/clip.f32                    
OK    corpus/clip-index.json             
OK    corpus/analytics/manifest.csv      
OK    corpus/analytics/knn-k20.csv       
OK    corpus/analytics/band.json         
OK    aesthetic/influences/              
OK    dist/studio/corpus.js              


## The loaders, exercised once each

Each of these is the *only* implementation of what it returns. If a notebook computes a kNN or a
dedupe or a tokenization itself, that is a bug in the notebook, not a shortcut.

In [3]:
m = k.load_manifest()
X, order = k.load_clip()
knn = k.load_knn()
b = k.band()

print(f"manifest        {len(m):>7,} rows x {m.shape[1]} columns")
print(f"clip.f32 (file) {k.load_clip_file()[0].shape[0]:>7,} rows")
print(f"clip (corpus)   {X.shape[0]:>7,} rows x {X.shape[1]} dims   <- 16 file rows name no manifest work")
print(f"kNN             {len(knn):>7,} rows = {knn.sha256.nunique():,} images x k={b['knnK']}")
print(f"duplicates      {m.is_duplicate_of_kept_row.sum():>7,} manifest rows share bytes with a kept row")
print(f"band            min {b['min']:.4f}  median {b['median']:.4f}  max {b['max']:.4f}  over {b['pairs']:,} pairs")

manifest         20,000 rows x 21 columns
clip.f32 (file)  19,807 rows
clip (corpus)    19,791 rows x 512 dims   <- 16 file rows name no manifest work
kNN             395,820 rows = 19,791 images x k=20
duplicates           98 manifest rows share bytes with a kept row
band            min 0.1555  median 0.6428  max 0.9685  over 1,124,250 pairs


## The text tower, over a subprocess

Reached through the TypeScript CLI on purpose. CLIP's tokenizer is byte-BPE with two special tokens
and a pool at the eos position; a port that gets any of that wrong still returns 512 finite numbers.

In [4]:
v = k.load_text_tower(["a sheet of laid paper", "a bronze reliquary"])
print("shape", v.shape, " unit length:", np.allclose(np.linalg.norm(v, axis=1), 1, atol=1e-4))
print(f"cosine between the two phrases: {float(v[0] @ v[1]):.4f}")

shape (2, 512)  unit length: True
cosine between the two phrases: 0.6514


## The one number to keep in mind

CLIP image embeddings sit in a narrow cone. Two corpus works picked at random are at cosine **0.64**.
So a cosine of 0.74 between an artwork and an influence set is not a relationship — it is slightly
above typical. Every similarity in these notebooks is therefore reported as a **percentile against
the band**, or against an explicit chance baseline, or not at all.

In [5]:
print(f"two random corpus works:      cosine {b['median']:.4f} (median of {b['pairs']:,} pairs)")
print(f"two random corpus works:      share a museum {k.museum_chance():.1%} of the time — not 1/3")
for mu in b["museums"]:
    print(f"  {mu['museum']}  {mu['works']:>6,}  {mu['share']:.1%}")
print()
print(k.Reading("k=20 neighbours that share a museum", b["sameMuseumShareAtK"], k.museum_chance()))

two random corpus works:      cosine 0.6428 (median of 1,124,250 pairs)
two random corpus works:      share a museum 39.0% of the time — not 1/3
  aic   6,788  34.3%
  cma   3,183  16.1%
  met   9,820  49.6%

k=20 neighbours that share a museum: 0.6634 vs 0.3897 — 1.70x chance


## Scratch

In [6]:
# yours